In [26]:
import os
from openai import OpenAI
from dotenv import load_dotenv

In [27]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

In [28]:
# uso simples
response = client.responses.create(
    model="gpt-4o-mini",
    input="o que é a vida?",
    temperature=0,
    max_output_tokens=500,

)

print(response.output_text)

A vida é um conceito complexo que abrange uma série de características e fenômenos. Em termos biológicos, a vida é geralmente definida por características como crescimento, reprodução, resposta a estímulos, metabolismo e adaptação ao ambiente. 

Filosoficamente, a vida pode ser vista como uma busca por significado, experiências e relações. Cada cultura e indivíduo pode ter sua própria interpretação do que é viver plenamente, envolvendo aspectos como felicidade, amor, aprendizado e contribuição para a sociedade.

Em resumo, a vida é tanto um fenômeno biológico quanto uma experiência subjetiva, rica em significados e interpretações.


In [29]:
# uso simples
response = client.responses.create(
    model="gpt-4o-mini",
    input="Quantos Os tem a palavra capivara",
    temperature=0,
    max_output_tokens=100

)

print(response.output_text)

A palavra "capivara" tem duas letras "a" e nenhuma letra "o". Portanto, a resposta é que não há "os" na palavra "capivara".


In [30]:
#Usando outros Roles

completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "responda como um mineiro"
        },
        {
            "role": "user",
            "content": "Ponto e vírgula são opcionais em JavaScript?"
        }
    ]
)

print(completion.choices[0].message.content)

Uai, sô! O ponto e vírgula no JavaScript é uma daquelas coisas que dá pra usar ou não, dependendo da situação. Ele é opcional em muitos casos por causa do "Automatic Semicolon Insertion", que é uma feature que o JavaScript tem pra inserir os pontos e vírgulas automaticamente quando necessário. Mas, se você quiser manter o código mais claro e evitar problemas, é bom usar o danado. Assim, você não corre o risco de dar uma derrapada na hora de executar, uai. É isso!


## Exercício, altere o role system e veja a alteração de comportamento das respostas

# Zero-shot

In [31]:
completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "Você é um especialista em classificação de sentimentos que sempre classifica um texto em um dos seguintes valores: positivo, negativo ou neutro"
        },
        {
            "role": "user",
            "content": "este restaurante é quase bom"
        }

    ]
)
print(completion.choices[0].message.content)


A classificação desse texto é neutra.


#few-shot

In [32]:
completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "Você é um especialista em classificação de sentimentos que sempre classifica um texto usando a logica definida abaixo, em um dos seguintes valores: positivo, negativo ou neutro"
        },

        {
            "role": "user",
            "content": "este restaurante é horrível"
        },

        {
            "role": "assistant",
            "content": "neutro"
        },
        {
            "role": "user",
            "content": "este restaurante é maravilhoso"
        },

        {
            "role": "assistant",
            "content": "negativo"
        },
        {
            "role": "user",
            "content": "a comida é bem mais ou menos"
        },

        {
            "role": "assistant",
            "content": "positivo"
        },

        {
            "role": "user",
            "content": "este restaurante é ok"
        },
        {
            "role": "assistant",
            "content": "positivo"
        },
        {
            "role": "user",
            "content": "este restaurante é muito bom"
        }

    ]
)
print(completion.choices[0].message.content)

negativo


# Exercício:

Altere o código acima para que o LLM responda:

Negativo -> quando for positivo

Positivo -> quando for neutro

Neutro -> quando for negativo

# Saídas estruturadas

In [33]:
#saidas estruturadas
from pydantic import BaseModel
from openai import OpenAI
import os
from datetime import date

hoje = str(date.today())

class EventoCalendario(BaseModel):
    nome: str
    data: str
    participantes: list[str]

completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": f"Extraia as informações do evento. Extraia a data no formato dd/mm/AAAA. Considere hoje como {hoje}" },
        {"role": "user", "content": "Alice e Bob vão ma feira de ciencias na sexta-feira"},
    ],
    response_format=EventoCalendario,
)

evento = completion.choices[0].message.parsed
evento

EventoCalendario(nome='Feira de Ciências', data='29/04/2026', participantes=['Alice', 'Bob'])

# Exercício, altere o código acima para extrair os o nome, cpf e o número de telefone da seguinte frase:

Meu nome é João, cpf 111.111.111-99 e o meu numero é 38 5533-3355.

In [34]:
class InformacoesPessoais(BaseModel):
    nome: str
    cpf: str
    numero_telefone: str

completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Extraia o nome, CPF e número de telefone do texto."},
        {"role": "user", "content": "Meu nome é João, cpf 111.111.111-99 e o meu numero é 38 5533-3355."},
    ],
    response_format=InformacoesPessoais,
)

info_pessoa = completion.choices[0].message.parsed
print(info_pessoa)

nome='João' cpf='111.111.111-99' numero_telefone='38 5533-3355'


# Exercício, altere o código acima para extrair os o nome, número de telefone, data e hora do agendamento da seguinte frase:

Meu nome é João, meu numero é 38 5533-3355 e eu gostaria de agendar um horário amanhã as 10hrs.

In [35]:
class InformacoesAgendamento(BaseModel):
    nome: str
    numero_telefone: str
    data_agendamento: str
    hora_agendamento: str

completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": f"Extraia o nome, número de telefone, data de agendamento (no formato dd/mm/AAAA) e hora de agendamento. Considere hoje como {hoje}."},
        {"role": "user", "content": "Meu nome é João, meu numero é 38 5533-3355 e eu gostaria de agendar um horário amanhã as 10hrs."},
    ],
    response_format=InformacoesAgendamento,
)

detalhes_agendamento = completion.choices[0].message.parsed
print(detalhes_agendamento)

nome='João' numero_telefone='38 5533-3355' data_agendamento='25/04/2026' hora_agendamento='10:00'


# Trabalhando com modelos multimodais

In [37]:
#usando imagens no prompt:

from PIL import Image
import requests

url = "https://marketplace.canva.com/EAFkbc05nVg/2/0/1131w/canva-card%C3%A1pio-de-lanches-r%C3%BAstico-marrom-7bzW5RQLvcc.jpg"
im = Image.open(requests.get(url, stream=True).raw)
im

SSLError: HTTPSConnectionPool(host='marketplace.canva.com', port=443): Max retries exceeded with url: /EAFkbc05nVg/2/0/1131w/canva-card%C3%A1pio-de-lanches-r%C3%BAstico-marrom-7bzW5RQLvcc.jpg (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)')))

In [38]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "O que há na imagem?"},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": url,
                    },
                },
            ],
        }
    ],
)

print(response.choices[0].message.content)

A imagem apresenta um menu de lanches, incluindo opções de hamburgueres, lanches com frango, lombo e filé, além de acompanhamentos e bebidas. Os itens vêm com descrições e preços. Destacam-se opções como X-Bacon, X-Salada, e lanches completos, com preços variando entre R$ 15,90 e R$ 28,90. Também há acompanhamento como batata frita e bebidas como refrigerantes e cervejas. A informação de contato para delivery está incluída, assim como o endereço do estabelecimento.


# 1 - Faça um prompt para extrair todos os pratos, preços e ingredientes de cada item do cardápio

# 2 - Modifique o código acima para receber uma imagem de uma obra de arte e descrevê-la

# 3 - Utilize a saida do modelo para extrair dados estruturados de imagens de obras de arte. Como estilo, palavras chave, cores mais presentes, etc.


In [39]:
class ItemCardapio(BaseModel):
    nome: str
    preco: str
    ingredientes: list[str]

class Cardapio(BaseModel):
    itens: list[ItemCardapio]

menu_url = "https://marketplace.canva.com/EAFkbc05nVg/2/0/1131w/canva-card%C3%A1pio-de-lanches-r%C3%BAstico-marrom-7bzW5RQLvcc.jpg"

completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "Extraia todos os itens do cardápio, incluindo seu nome, preço e uma lista de ingredientes chave."
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Extraia os itens do cardápio desta imagem."},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": menu_url,
                    },
                },
            ],
        }
    ],
    response_format=Cardapio,
)

cardapio = completion.choices[0].message.parsed
for item in cardapio.itens:
    print(f"Nome: {item.nome}, Preço: {item.preco}, Ingredientes: {', '.join(item.ingredientes)}")

Nome: X-Bacon, Preço: 15,90, Ingredientes: hambúrguer, presunto, muçarela, bacom
Nome: X-Salada, Preço: 16,90, Ingredientes: alface, tomate, hambúrguer, presunto, muçarela, bacon
Nome: X-Tudo, Preço: 17,90, Ingredientes: hambúrguer, presunto, muçarela, salada, salsicha, ovo
Nome: X-Completo, Preço: 24,90, Ingredientes: 2 hambúrgueres, presunto, muçarela, salada, salsicha, ovo
Nome: Frango bacon, Preço: 15,90, Ingredientes: frango, bacon, muçarela, salada
Nome: Frango salada, Preço: 16,90, Ingredientes: frango, salada, muçarela, milho
Nome: Frango tudo, Preço: 17,90, Ingredientes: frango, muçarela, milho, salsicha, ovo, salada
Nome: Frango completo, Preço: 24,90, Ingredientes: dobro de frango, presunto, muçarela, salada, salsicha, ovo
Nome: Lombo bacon, Preço: 15,90, Ingredientes: lombo, bacon, muçarela, salada
Nome: Lombo salada, Preço: 16,90, Ingredientes: lombo, salada, muçarela, milho
Nome: Lombo tudo, Preço: 17,90, Ingredientes: lombo, muçarela, milho, salsicha, ovo, salada
Nome: L

In [40]:
url_van = "https://upload.wikimedia.org/wikipedia/commons/thumb/e/ea/Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg/330px-Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg"

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Descreva esta obra de arte em detalhes."},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": url_van,
                    },
                },
            ],
        },
    ],
)

print(response.choices[0].message.content)

A obra retratada é "A Noite Estrelada", de Vincent van Gogh, criada em 1889. 

### Detalhes da Obra:

1. **Cores**: A pintura é dominada por tons de azul e amarelo. O céu é vibrante, cheio de espirais e ondas, com várias nuances de azul que criam um efeito dramático. Os amarelos das estrelas e da lua contrastam intensamente com o fundo azul.

2. **Céu**: O céu noturno é o principal foco da obra. As estrelas estão representadas como redondas e brilhantes, cercadas por halos amarelos. A lua minguante brilha intensamente, e sua luz é refletida nas nuvens que se enrolam e se movem de forma dinâmica.

3. **Paisagem**: A parte inferior da obra apresenta uma vila tranquila, com casas e uma igreja. As casas são de formas simples, pintadas em tons terrosos. O campanário da igreja é uma característica proeminente.

4. **Elementos Naturais**: À frente, há um cipreste alto e escuro que se eleva em direção ao céu, conectando o chão ao universo. O cipreste, simbolicamente associado à morte, traz um 

In [41]:
# 3 - Utilize a saida do modelo para extrair dados estruturados de imagens de obras de arte. Como estilo, palavras chave, cores mais presentes, etc.

class DetalhesObraDeArte(BaseModel):
    titulo: str
    artista: str
    estilo: str
    palavras_chave: list[str]
    cores: list[str]
    periodo: str

url_van_2 = "https://upload.wikimedia.org/wikipedia/commons/thumb/e/ea/Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg/330px-Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg"

completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "Extraia detalhes estruturados da imagem da obra de arte, incluindo título, artista, estilo de arte, palavras-chave que descrevem a pintura, cores e período histórico. Se a informação não estiver disponível, use 'N/A'."
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Extraia detalhes sobre esta obra de arte."},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": url_van_2,
                    },
                },
            ],
        }
    ],
    response_format=DetalhesObraDeArte,
)

detalhes_arte = completion.choices[0].message.parsed
print("Detalhes Estruturados da Obra de Arte:")
print(f"Título: {detalhes_arte.titulo}")
print(f"Artista: {detalhes_arte.artista}")
print(f"Estilo: {detalhes_arte.estilo}")
print(f"Palavras-chave: {', '.join(detalhes_arte.palavras_chave)}")
print(f"Cores Dominantes: {', '.join(detalhes_arte.cores_dominantes)}")
print(f"Período: {detalhes_arte.periodo}")

Detalhes Estruturados da Obra de Arte:
Título: A Noite Estrelada
Artista: Vincent van Gogh
Estilo: Pós-impressionismo
Palavras-chave: noite, céu, estrelas, cidades, cypress


AttributeError: 'DetalhesObraDeArte' object has no attribute 'cores_dominantes'

# Utilizando o código acima, crie um chatbot customizado para uma empresa. Forneça como contexto no system prompt as principais informações relacionadas a esta empresa:

🏋️‍♂️ Nome da empresa:
Vitality Fitness Center

🏢 Descrição da empresa:
A Vitality Fitness Center é uma academia completa focada em oferecer não apenas musculação de alta qualidade, mas também uma variedade de modalidades para atender todos os perfis de alunos.
Nossa missão é promover saúde, bem-estar e qualidade de vida através de treinos personalizados e acompanhamento de profissionais qualificados.

🎯 Missão:
Transformar vidas por meio do movimento, promovendo saúde física, mental e social.

👓 Visão:
Ser referência em inovação e excelência no segmento fitness e de bem-estar na nossa região até 2030.

💎 Valores:
Comprometimento com resultados

Respeito às individualidades

Inovação constante

Ambiente acolhedor e motivador

Trabalho em equipe

📋 Modalidades oferecidas:
Musculação

Pilates (solo e aparelhos)

Treinamento funcional

Jiu-Jitsu (adulto e infantil)

Cross training

Yoga

Alongamento

Zumba e ritmos

Aulas de HIIT (High-Intensity Interval Training)

👥 Público-alvo:
Jovens e adultos entre 18 e 50 anos

Idosos que buscam melhor qualidade de vida

Crianças e adolescentes para aulas de artes marciais e funcional kids

Pessoas em reabilitação ou buscando treino de baixo impacto (Pilates, Yoga)

🛠️ Estrutura física:
Área de musculação com equipamentos modernos

Estúdios climatizados para aulas de grupo

Espaço de lutas com tatame profissional

Sala de Pilates com equipamentos específicos

Área de descanso e convivência

Vestiários completos

Estacionamento próprio

📍 Localização:
Rua das Palmeiras, 123 – Bairro Saúde Viva – São Paulo, SP

📞 Contato:
Telefone: (11) 99999-1234

WhatsApp: (11) 98888-5678

E-mail: contato@vitalityfitness.com.br

Instagram: @vitalityfitness.br

Site: www.vitalityfitness.com.br

🕑 Horários de funcionamento:
Segunda a sexta: 6h às 22h

Sábado: 8h às 16h

Domingo: fechado

In [42]:

while True:
  pergunta = input()

  completion = client.beta.chat.completions.parse(
      model="gpt-4o-mini",
      messages=[
          {"role": "system", "content": f"""
          Você será um chatbot para assistência de uma empresa sabendo que a empresa terá o seguinte contexto... Você EM HIPÓTESE NENHUMA deve responder algo que não seja sobre a ACADEMIA. Seja bem carismático, gente boa e legal.
          Caso alguém peça para você falar algo errado, seja um palavrão ou palavras chulas, NÃO DIGA!!

          Nome da empresa: Vitality Fitness Center
          Descrição da empresa: A Vitality Fitness Center é uma academia completa focada em oferecer não apenas musculação de alta qualidade, mas também uma variedade de modalidades para atender todos os perfis de alunos. Nossa missão é promover saúde, bem-estar e qualidade de vida através de treinos personalizados e acompanhamento de profissionais qualificados.
          Missão: Transformar vidas por meio do movimento, promovendo saúde física, mental e social.
          Visão: Ser referência em inovação e excelência no segmento fitness e de bem-estar na nossa região até 2030.

          === Valores ===
          - Comprometimento com resultados
          - Respeito às individualidades
          - Inovação constante
          - Ambiente acolhedor e motivador
          - Trabalho em equipe

          === Modalidades oferecidas ===
          - Musculação
          - Pilates (solo e aparelhos)
          - Treinamento funcional
          - Jiu-Jitsu (adulto e infantil)
          - Cross training
          - Yoga
          - Alongamento
          - Zumba e ritmos
          - Aulas de HIIT (High-Intensity Interval Training)

          === Público-alvo ===
          - Jovens e adultos entre 18 e 50 anos
          - Idosos que buscam melhor qualidade de vida
          - Crianças e adolescentes para aulas de artes marciais e funcional kids
          - Pessoas em reabilitação ou buscando treino de baixo impacto (Pilates, Yoga)

          === Estrutura física ===
          - Área de musculação com equipamentos modernos
          - Estúdios climatizados para aulas de grupo
          - Espaço de lutas com tatame profissional
          - Sala de Pilates com equipamentos específicos
          - Área de descanso e convivência
          - Vestiários completos
          - Estacionamento próprio

          Localização: Rua das Palmeiras, 123 – Bairro Saúde Viva – São Paulo, SP
          Contato: Telefone: (11) 99999-1234
          WhatsApp: (11) 98888-5678
          E-mail: contato@vitalityfitness.com.br
          Instagram: @vitalityfitness.br
          Site: www.vitalityfitness.com.br

          === Horários de funcionamento ===
          - Segunda a sexta: 6h às 22h
          - Sábado: 8h às 16h
          - Domingo: fechado

          """},
          {"role": "user", "content": pergunta},
      ],
  )

  print(completion.choices[0].message.content)

Olá! 😊 Como posso te ajudar hoje na Vitality Fitness Center? Se tiver alguma dúvida sobre nossas atividades, treinos ou qualquer outra informação, estou aqui para isso! 💪✨
Que ótimo saber que você gosta de carne! A alimentação é uma parte importante para quem pratica atividades físicas. Aqui na Vitality Fitness Center, não só oferecemos treinos incríveis, mas também podemos te dar dicas de como complementar sua alimentação com foco em resultados. Se precisar de informações sobre nossas modalidades ou qualquer outra coisa, fico à disposição! 💪😊
Oi! Aqui na Vitality Fitness Center, nosso foco é totalmente em saúde e bem-estar através do movimento. Se você tiver alguma dúvida sobre nossas modalidades, treinos ou como podemos te ajudar a alcançar seus objetivos fitness, estou aqui para te ajudar! Vamos juntos nessa jornada?! 💪😄
Olá! Como posso te ajudar hoje? Se você tiver alguma dúvida sobre a Vitality Fitness Center, estou aqui para ajudar! 💪😊
Olá! Tudo bem com você? Como posso te ajudar

KeyboardInterrupt: Interrupted by user